# LLaMA2-7B Configurable BiE G16 Perplexity

This notebook evaluates a **configurable BiE W/A, Group-16** fake-quantized LLaMA2-7B model on WikiText-2. Set `BiEConfig.mantissa_bits` to choose the private format (`3` gives BiE4/1S3M, `4` gives BiE5/1S4M, and so on). Transformer Linear weights and input activations use two shared E5 exponents per block; `lm_head` remains FP16.

The requested outlier rule is the Oiso explicit-outlier definition **T = mean(X) + 3 * std(X)**, followed by **abs(x) > T**. It is intentionally recorded as a BiE signed-3sigma research variant rather than the original BiE paper's offline threshold-search flow or Oiso's final MSE-optimized threshold flow. Decoder `nn.Linear` operations are quantized while `lm_head` remains FP16.

In [ ]:
%pip install -q "transformers==5.13.1" "datasets==4.0.0" accelerate sentencepiece tqdm

In [ ]:
import gc
import json
import math
import os
import platform
import time
from dataclasses import asdict, dataclass
from getpass import getpass
from pathlib import Path

import datasets
import torch
import torch.nn as nn
import torch.nn.functional as F
import transformers
from datasets import load_dataset
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_ID = "meta-llama/Llama-2-7b-hf"
DATASET_ID = "Salesforce/wikitext"
DATASET_CONFIG = "wikitext-2-raw-v1"
SPLIT = "test"
CONTEXT_LENGTH = 2048
STRIDE = 2048
DROP_REMAINDER = True
EVALUATION_PROTOCOL = "non_overlapping_2048_drop_remainder"
BASELINE_PPL = 5.472103118896484


@dataclass(frozen=True)
class BiEConfig:
    block_size: int = 16
    shared_exponent_bits: int = 5
    mantissa_bits: int = 4  # BiE5 private value = 1 sign bit + 4 magnitude bits.
    rounding: str = "nearest"
    threshold_method: str = "mean_plus_sigma_k_std"
    sigma_k: float = 3.0
    weight_chunk_rows: int = 128
    activation_chunk_rows: int = 2048
    quantize_lm_head: bool = False

    def validate(self):
        if self.block_size != 16:
            raise ValueError("This notebook is fixed to Group-16.")
        if self.shared_exponent_bits != 5:
            raise ValueError("This notebook is fixed to two E5 shared exponents.")
        if self.mantissa_bits <= 0:
            raise ValueError("mantissa_bits must be positive.")
        if self.rounding != "nearest":
            raise ValueError("This notebook is fixed to nearest rounding.")
        if self.threshold_method != "mean_plus_sigma_k_std":
            raise ValueError("Unexpected threshold method.")
        if not math.isfinite(self.sigma_k) or self.sigma_k < 0:
            raise ValueError("sigma_k must be finite and non-negative.")
        if min(self.weight_chunk_rows, self.activation_chunk_rows) <= 0:
            raise ValueError("Chunk sizes must be positive.")


BIE = BiEConfig()
BIE.validate()
PRIVATE_VALUE_BITS = 1 + BIE.mantissa_bits
OUTPUT_PATH = Path(
    f"bie{PRIVATE_VALUE_BITS}-g{BIE.block_size}-signed-mu3sigma-no-lm-head-s2048.json"
)
EFFECTIVE_BITS_PER_VALUE = (
    PRIVATE_VALUE_BITS
    + 1  # Per-element normal/outlier type bit.
    + 2 * BIE.shared_exponent_bits / BIE.block_size
)

torch.manual_seed(0)
torch.backends.cuda.matmul.allow_tf32 = False

print(f"BiE config: {BIE}")
print(f"Effective storage: {EFFECTIVE_BITS_PER_VALUE:.3f} bits/value (threshold metadata excluded)")
print(f"Output: {OUTPUT_PATH.resolve()}")
if not torch.cuda.is_available():
    print("CUDA is unavailable: smoke tests can run on CPU; full PPL evaluation cannot.")

## BiE encoder

The threshold is computed once over the complete unpadded tensor. Chunking is used only for memory control after the threshold is frozen. Every Group-16 block then derives one normal exponent and one outlier exponent.

In [ ]:
COUNT_NAMES = (
    "total_values",
    "outlier_values",
    "total_blocks",
    "outlier_blocks",
    "normal_only_blocks",
    "outlier_only_blocks",
    "mixed_blocks",
)


def _empty_counts(device):
    return torch.zeros(len(COUNT_NAMES), dtype=torch.int64, device=device)


@torch.no_grad()
def _signed_tensor_threshold(tensor, sigma_k, chunk_rows):
    width = tensor.shape[-1]
    flat = tensor.reshape(-1, width)
    if flat.numel() == 0:
        raise ValueError("Cannot compute a threshold for an empty tensor.")
    if chunk_rows <= 0:
        raise ValueError("chunk_rows must be positive.")

    running_count = 0
    running_mean = torch.zeros((), dtype=torch.float64, device=flat.device)
    running_m2 = torch.zeros((), dtype=torch.float64, device=flat.device)

    for start in range(0, flat.size(0), chunk_rows):
        end = min(start + chunk_rows, flat.size(0))
        values = flat[start:end].float()
        chunk_count = values.numel()
        chunk_var, chunk_mean = torch.var_mean(values, unbiased=False)
        chunk_mean = chunk_mean.to(torch.float64)
        chunk_var = chunk_var.to(torch.float64)

        if running_count == 0:
            running_mean = chunk_mean
            running_m2 = chunk_var * chunk_count
            running_count = chunk_count
            continue

        combined_count = running_count + chunk_count
        delta = chunk_mean - running_mean
        running_mean = running_mean + delta * (chunk_count / combined_count)
        running_m2 = (
            running_m2
            + chunk_var * chunk_count
            + delta.square() * running_count * chunk_count / combined_count
        )
        running_count = combined_count

    variance = (running_m2 / running_count).clamp_min(0.0)
    threshold = running_mean + sigma_k * torch.sqrt(variance)
    return threshold.to(torch.float32)


def _shared_exponent(max_abs, present, config):
    safe_max = max_abs.clamp_min(torch.finfo(torch.float32).tiny)
    exponent = torch.floor(torch.log2(safe_max))
    exp_min = -(1 << (config.shared_exponent_bits - 1))
    exp_max = (1 << (config.shared_exponent_bits - 1)) - 1
    exponent = exponent.clamp(exp_min, exp_max)
    return torch.where(present, exponent, torch.zeros_like(exponent))


@torch.no_grad()
def _quantize_bie_rows_with_threshold(rows, threshold, config):
    original_shape = rows.shape
    width = original_shape[-1]
    flat = rows.reshape(-1, width).float()
    padding = (-width) % config.block_size

    valid = torch.ones_like(flat, dtype=torch.bool)
    if padding:
        flat = F.pad(flat, (0, padding))
        valid = F.pad(valid, (0, padding), value=False)

    padded_width = flat.size(1)
    blocks = flat.reshape(flat.size(0), -1, config.block_size)
    valid_blocks = valid.reshape_as(blocks)
    magnitude = blocks.abs()

    outlier = valid_blocks & (magnitude > threshold)
    normal = valid_blocks & ~outlier
    normal_present = normal.any(dim=-1, keepdim=True)
    outlier_present = outlier.any(dim=-1, keepdim=True)

    normal_max = torch.where(normal, magnitude, 0.0).amax(dim=-1, keepdim=True)
    outlier_max = torch.where(outlier, magnitude, 0.0).amax(dim=-1, keepdim=True)
    normal_exp = _shared_exponent(normal_max, normal_present, config)
    outlier_exp = _shared_exponent(outlier_max, outlier_present, config)

    normal_exp = torch.where(
        ~normal_present & outlier_present, outlier_exp, normal_exp
    )
    outlier_exp = torch.where(
        ~outlier_present & normal_present, normal_exp, outlier_exp
    )
    selected_exp = torch.where(outlier, outlier_exp, normal_exp)

    step = torch.pow(2.0, selected_exp - (config.mantissa_bits - 1))
    mantissa = blocks / step
    mantissa = torch.round(mantissa)
    mantissa_max = (1 << config.mantissa_bits) - 1
    mantissa = mantissa.clamp(-mantissa_max, mantissa_max)

    dequantized = (mantissa * step).reshape(flat.size(0), padded_width)
    dequantized = dequantized[:, :width].reshape(original_shape).to(rows.dtype)

    real_block = valid_blocks.any(dim=-1, keepdim=True)
    normal_only = real_block & normal_present & ~outlier_present
    outlier_only = real_block & outlier_present & ~normal_present
    mixed = real_block & normal_present & outlier_present
    counts = torch.stack(
        (
            torch.tensor(rows.numel(), dtype=torch.int64, device=rows.device),
            outlier.sum(dtype=torch.int64),
            real_block.sum(dtype=torch.int64),
            outlier_present.sum(dtype=torch.int64),
            normal_only.sum(dtype=torch.int64),
            outlier_only.sum(dtype=torch.int64),
            mixed.sum(dtype=torch.int64),
        )
    )
    return dequantized, counts


@torch.no_grad()
def quantize_bie(tensor, config, chunk_rows):
    width = tensor.shape[-1]
    flat = tensor.reshape(-1, width)
    threshold = _signed_tensor_threshold(flat, config.sigma_k, chunk_rows)
    output = torch.empty_like(flat)
    counts = _empty_counts(flat.device)

    for start in range(0, flat.size(0), chunk_rows):
        end = min(start + chunk_rows, flat.size(0))
        quantized, chunk_counts = _quantize_bie_rows_with_threshold(
            flat[start:end], threshold, config
        )
        output[start:end] = quantized
        counts.add_(chunk_counts)

    return output.reshape_as(tensor), threshold, counts


@torch.no_grad()
def quantize_weight_in_place(weight, config):
    threshold = _signed_tensor_threshold(
        weight, config.sigma_k, config.weight_chunk_rows
    )
    counts = _empty_counts(weight.device)

    for start in range(0, weight.size(0), config.weight_chunk_rows):
        end = min(start + config.weight_chunk_rows, weight.size(0))
        quantized, chunk_counts = _quantize_bie_rows_with_threshold(
            weight[start:end], threshold, config
        )
        weight[start:end].copy_(quantized)
        counts.add_(chunk_counts)

    return threshold, counts

## Linear replacement and self-describing statistics

Weight statistics are captured once during model conversion. Activation counters stay on the GPU during evaluation to avoid a CPU synchronization on every Linear call.

In [ ]:
def _rate(numerator, denominator):
    return {
        "numerator": int(numerator),
        "denominator": int(denominator),
        "rate": None if denominator == 0 else numerator / denominator,
    }


def _counts_to_dict(counts):
    values = counts.detach().cpu().tolist()
    return {name: int(value) for name, value in zip(COUNT_NAMES, values)}


def _summarize_counts(counts):
    return {
        "counts": counts,
        "rates": {
            "outlier_value_rate": _rate(
                counts["outlier_values"], counts["total_values"]
            ),
            "secondary_exponent_value_rate": _rate(
                counts["outlier_values"], counts["total_values"]
            ),
            "outlier_block_rate": _rate(
                counts["outlier_blocks"], counts["total_blocks"]
            ),
            "normal_only_block_rate": _rate(
                counts["normal_only_blocks"], counts["total_blocks"]
            ),
            "outlier_only_block_rate": _rate(
                counts["outlier_only_blocks"], counts["total_blocks"]
            ),
            "mixed_block_rate": _rate(
                counts["mixed_blocks"], counts["total_blocks"]
            ),
        },
    }


class BiELinear(nn.Module):
    def __init__(self, linear, config, weight_threshold, weight_counts):
        super().__init__()
        self.linear = linear
        self.config = config
        self.weight_threshold = float(weight_threshold.detach().cpu().item())
        self.weight_counts = _counts_to_dict(weight_counts)

        device = linear.weight.device
        self.register_buffer(
            "_activation_counts",
            _empty_counts(device),
            persistent=False,
        )
        self.register_buffer(
            "_activation_calls",
            torch.zeros((), dtype=torch.int64, device=device),
            persistent=False,
        )
        self.register_buffer(
            "_activation_threshold_sum",
            torch.zeros((), dtype=torch.float64, device=device),
            persistent=False,
        )
        self.register_buffer(
            "_activation_threshold_min",
            torch.tensor(float("inf"), dtype=torch.float64, device=device),
            persistent=False,
        )
        self.register_buffer(
            "_activation_threshold_max",
            torch.tensor(float("-inf"), dtype=torch.float64, device=device),
            persistent=False,
        )

    def forward(self, x):
        x_bie, threshold, counts = quantize_bie(
            x, self.config, self.config.activation_chunk_rows
        )
        threshold64 = threshold.to(torch.float64)
        self._activation_counts.add_(counts)
        self._activation_calls.add_(1)
        self._activation_threshold_sum.add_(threshold64)
        self._activation_threshold_min.copy_(
            torch.minimum(self._activation_threshold_min, threshold64)
        )
        self._activation_threshold_max.copy_(
            torch.maximum(self._activation_threshold_max, threshold64)
        )
        return F.linear(
            x_bie, self.linear.weight, self.linear.bias
        ).to(torch.float16)

    def export_stats(self):
        calls = int(self._activation_calls.detach().cpu().item())
        if calls:
            threshold_summary = {
                "count": calls,
                "mean": float(
                    self._activation_threshold_sum.detach().cpu().item() / calls
                ),
                "min": float(self._activation_threshold_min.detach().cpu().item()),
                "max": float(self._activation_threshold_max.detach().cpu().item()),
            }
        else:
            threshold_summary = {
                "count": 0,
                "mean": None,
                "min": None,
                "max": None,
            }

        return {
            "weight": {
                "threshold": self.weight_threshold,
                **_summarize_counts(self.weight_counts),
            },
            "activation": {
                "threshold_summary": threshold_summary,
                **_summarize_counts(_counts_to_dict(self._activation_counts)),
            },
        }


def replace_linear_layers(module, config, prefix=""):
    replaced = {}

    for name, child in list(module.named_children()):
        full_name = f"{prefix}.{name}" if prefix else name

        if isinstance(child, nn.Linear):
            if full_name == "lm_head" and not config.quantize_lm_head:
                continue
            weight_threshold, weight_counts = quantize_weight_in_place(
                child.weight, config
            )
            wrapper = BiELinear(
                child, config, weight_threshold, weight_counts
            )
            setattr(module, name, wrapper)
            replaced[full_name] = wrapper
        else:
            replaced.update(
                replace_linear_layers(child, config, full_name)
            )

    return replaced


def _add_counts(total, current):
    for name in COUNT_NAMES:
        total[name] += current[name]


def export_quantization_stats(replaced):
    weight_totals = {name: 0 for name in COUNT_NAMES}
    activation_totals = {name: 0 for name in COUNT_NAMES}
    weight_thresholds = []
    activation_threshold_sum = 0.0
    activation_threshold_count = 0
    activation_threshold_min = float("inf")
    activation_threshold_max = float("-inf")
    layers = []

    for name in sorted(replaced):
        layer_stats = replaced[name].export_stats()
        layers.append({"layer_name": name, **layer_stats})
        _add_counts(weight_totals, layer_stats["weight"]["counts"])
        _add_counts(activation_totals, layer_stats["activation"]["counts"])
        weight_thresholds.append(layer_stats["weight"]["threshold"])

        summary = layer_stats["activation"]["threshold_summary"]
        if summary["count"]:
            activation_threshold_sum += summary["mean"] * summary["count"]
            activation_threshold_count += summary["count"]
            activation_threshold_min = min(
                activation_threshold_min, summary["min"]
            )
            activation_threshold_max = max(
                activation_threshold_max, summary["max"]
            )

    weight_threshold_summary = {
        "count": len(weight_thresholds),
        "mean": sum(weight_thresholds) / len(weight_thresholds),
        "min": min(weight_thresholds),
        "max": max(weight_thresholds),
    }
    activation_threshold_summary = {
        "count": activation_threshold_count,
        "mean": (
            None
            if activation_threshold_count == 0
            else activation_threshold_sum / activation_threshold_count
        ),
        "min": (
            None if activation_threshold_count == 0 else activation_threshold_min
        ),
        "max": (
            None if activation_threshold_count == 0 else activation_threshold_max
        ),
    }

    return {
        "aggregate": {
            "weight": {
                "threshold_summary": weight_threshold_summary,
                **_summarize_counts(weight_totals),
            },
            "activation": {
                "threshold_summary": activation_threshold_summary,
                **_summarize_counts(activation_totals),
            },
        },
        "layers": layers,
    }

## Synthetic checks

These checks cover zero blocks, missing partitions, mixed partitions, chunk invariance, and both weight/activation paths through the Linear wrapper.

In [ ]:
def _quantize_single_exponent_reference(rows, config):
    original_shape = rows.shape
    width = original_shape[-1]
    flat = rows.reshape(-1, width).float()
    padding = (-width) % config.block_size
    if padding:
        flat = F.pad(flat, (0, padding))

    padded_width = flat.size(1)
    blocks = flat.reshape(flat.size(0), -1, config.block_size)
    max_abs = blocks.abs().amax(dim=-1, keepdim=True)
    exponent = _shared_exponent(
        max_abs, max_abs != 0, config
    )
    step = torch.pow(2.0, exponent - (config.mantissa_bits - 1))
    mantissa_max = (1 << config.mantissa_bits) - 1
    mantissa = torch.round(blocks / step).clamp(
        -mantissa_max, mantissa_max
    )
    output = (mantissa * step).reshape(flat.size(0), padded_width)
    return output[:, :width].reshape(original_shape).to(rows.dtype)


_test_device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

_zero = torch.zeros((2, 16), dtype=torch.float32, device=_test_device)
_zero_q, _zero_t, _zero_counts = quantize_bie(_zero, BIE, chunk_rows=1)
_zero_stats = _counts_to_dict(_zero_counts)
assert torch.equal(_zero_q, _zero)
assert _zero_t.item() == 0.0
assert _zero_stats["normal_only_blocks"] == 2
assert _zero_stats["outlier_values"] == 0

_mixed = torch.zeros((1, 16), dtype=torch.float32, device=_test_device)
_mixed[0, -1] = -100.0
_mixed_q, _mixed_t, _mixed_counts = quantize_bie(
    _mixed, BIE, chunk_rows=1
)
_mixed_stats = _counts_to_dict(_mixed_counts)
_mixed_expected_t = (
    _mixed.mean()
    + BIE.sigma_k * _mixed.std(unbiased=False)
)
assert torch.allclose(_mixed_t, _mixed_expected_t, rtol=1e-6, atol=1e-6)
_mixed_abs_t = (
    _mixed.abs().mean()
    + BIE.sigma_k * _mixed.abs().std(unbiased=False)
)
assert not torch.allclose(_mixed_t, _mixed_abs_t, rtol=1e-6, atol=1e-6)
assert 0.0 < _mixed_t.item() < 100.0
assert _mixed_stats["outlier_values"] == 1
assert _mixed_stats["mixed_blocks"] == 1
assert _mixed_q[0, -1] != 0

_all_outlier = torch.ones(
    (1, 16), dtype=torch.float32, device=_test_device
)
_, _all_outlier_counts = _quantize_bie_rows_with_threshold(
    _all_outlier, torch.tensor(0.5, device=_test_device), BIE
)
_all_outlier_stats = _counts_to_dict(_all_outlier_counts)
assert _all_outlier_stats["outlier_only_blocks"] == 1
assert _all_outlier_stats["outlier_values"] == 16

_single_path = torch.ones(
    (2, 16), dtype=torch.float32, device=_test_device
)
_single_q, _, _single_counts = quantize_bie(
    _single_path, BIE, chunk_rows=2
)
_reference_q = _quantize_single_exponent_reference(_single_path, BIE)
assert torch.equal(_single_q, _reference_q)
assert _counts_to_dict(_single_counts)["outlier_values"] == 0

torch.manual_seed(7)
_chunk_input = torch.randn(
    (5, 32), dtype=torch.float32, device=_test_device
)
_chunk_q1, _chunk_t1, _chunk_c1 = quantize_bie(
    _chunk_input, BIE, chunk_rows=1
)
_chunk_q5, _chunk_t5, _chunk_c5 = quantize_bie(
    _chunk_input, BIE, chunk_rows=5
)
assert torch.allclose(_chunk_t1, _chunk_t5, rtol=1e-6, atol=1e-6)
assert torch.equal(_chunk_q1, _chunk_q5)
assert torch.equal(_chunk_c1, _chunk_c5)

_toy_dtype = torch.float16 if _test_device.type == "cuda" else torch.float32
_toy = nn.Sequential(
    nn.Linear(16, 8, bias=False, device=_test_device, dtype=_toy_dtype)
)
_toy_layers = replace_linear_layers(_toy, BIE)
assert len(_toy_layers) == 1
_toy_x = torch.randn(
    (2, 16), device=_test_device, dtype=_toy_dtype
)
_toy_y = _toy(_toy_x)
_toy_stats = next(iter(_toy_layers.values())).export_stats()
assert _toy_y.shape == (2, 8)
assert torch.isfinite(_toy_y).all()
assert _toy_stats["weight"]["counts"]["total_values"] == 8 * 16
assert _toy_stats["activation"]["counts"]["total_values"] == 2 * 16
assert _toy_stats["activation"]["threshold_summary"]["count"] == 1

del _zero, _mixed, _all_outlier, _single_path, _chunk_input, _toy, _toy_x, _toy_y
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print(f"Synthetic BiE{PRIVATE_VALUE_BITS} checks passed.")

## Dataset and tokenizer

In [ ]:
token = os.getenv("HF_TOKEN")
if not token:
    try:
        from google.colab import userdata
        token = userdata.get("HF_TOKEN")
    except Exception:
        token = None
if not token:
    token = getpass("HF_TOKEN: ")

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, token=token)
dataset = load_dataset(DATASET_ID, DATASET_CONFIG, split=SPLIT)
text = "\n\n".join(dataset["text"])
input_ids = tokenizer(text, return_tensors="pt").input_ids
print(f"WikiText-2 {SPLIT} tokens: {input_ids.numel():,}")

## Non-overlapping perplexity evaluation

In [ ]:
@torch.inference_mode()
def evaluate_perplexity(model, input_ids, context_length, stride, drop_remainder):
    if stride != context_length:
        raise ValueError("Non-overlapping evaluation requires stride == context_length.")
    if not drop_remainder:
        raise ValueError("This evaluation requires drop_remainder=True.")
    if context_length > model.config.max_position_embeddings:
        raise ValueError("context_length exceeds the model context window.")

    device = next(model.parameters()).device
    sequence_length = input_ids.size(1)
    usable_length = sequence_length // context_length * context_length
    dropped_tokens = sequence_length - usable_length
    if usable_length == 0:
        raise ValueError("Input does not contain a complete context block.")

    total_nll = 0.0
    total_loss_tokens = 0
    total_blocks = usable_length // context_length

    torch.cuda.reset_peak_memory_stats(device)
    torch.cuda.synchronize(device)
    start_time = time.perf_counter()

    for begin in tqdm(
        range(0, usable_length, stride),
        total=total_blocks,
        desc=f"Evaluating BiE{PRIVATE_VALUE_BITS} G{BIE.block_size}",
    ):
        end = begin + context_length
        batch = input_ids[:, begin:end].to(device)
        labels = batch.clone()
        loss = model(batch, labels=labels, use_cache=False).loss
        loss_tokens = labels[:, 1:].numel()
        total_nll += loss.float().item() * loss_tokens
        total_loss_tokens += loss_tokens

    torch.cuda.synchronize(device)
    elapsed_seconds = time.perf_counter() - start_time
    mean_nll = total_nll / total_loss_tokens

    return {
        "mean_nll": mean_nll,
        "perplexity": float(torch.exp(torch.tensor(mean_nll))),
        "source_input_tokens": sequence_length,
        "used_input_tokens": usable_length,
        "dropped_input_tokens": dropped_tokens,
        "evaluated_blocks": total_blocks,
        "evaluated_tokens": total_loss_tokens,
        "elapsed_seconds": elapsed_seconds,
        "tokens_per_second": total_loss_tokens / elapsed_seconds,
        "peak_gpu_memory_gib": (
            torch.cuda.max_memory_allocated(device) / 2**30
        ),
    }

## Load and quantize LLaMA2-7B

Every selected Transformer Linear weight tensor receives its own tensor-wide signed-value mu+3sigma threshold before Group-16 encoding. Activations use one tensor-wide signed-value threshold for each selected Linear call; classification remains `abs(x) > threshold`, and `lm_head` remains FP16.

In [ ]:
if not torch.cuda.is_available():
    raise RuntimeError("The full PPL evaluation requires an NVIDIA CUDA GPU.")

torch.manual_seed(0)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    dtype=torch.float16,
    device_map=0,
    attn_implementation="eager",
    token=token,
)
model.eval()
model.config.use_cache = False

bie_layers = replace_linear_layers(model, BIE)
if not bie_layers:
    raise RuntimeError("No nn.Linear layers were replaced.")
if "lm_head" in bie_layers or not isinstance(model.lm_head, nn.Linear):
    raise RuntimeError("lm_head must remain an unwrapped FP16 nn.Linear.")
if len(bie_layers) != 224:
    raise RuntimeError(f"Expected 224 quantized decoder Linear layers, got {len(bie_layers)}.")
torch.cuda.empty_cache()

parameter_dtypes = {
    parameter.dtype
    for parameter in model.parameters()
    if parameter.is_floating_point()
}
parameter_devices = {
    parameter.device.type for parameter in model.parameters()
}
assert parameter_dtypes == {torch.float16}, parameter_dtypes
assert parameter_devices == {"cuda"}, parameter_devices

print(f"BiE-quantized Linear layers: {len(bie_layers)}")
print("lm_head: FP16 (not BiE-quantized)")
print(f"BiE format: 1S{BIE.mantissa_bits}M + two E{BIE.shared_exponent_bits} + type bit")
print(f"Threshold: mean(X) + {BIE.sigma_k:g} * std(X); outlier iff abs(X) > threshold")

## Run, validate, and save JSON

In [ ]:
metrics = evaluate_perplexity(
    model,
    input_ids,
    CONTEXT_LENGTH,
    STRIDE,
    DROP_REMAINDER,
)
quantization_stats = export_quantization_stats(bie_layers)

for domain in ("weight", "activation"):
    counts = quantization_stats["aggregate"][domain]["counts"]
    if counts["total_blocks"] != (
        counts["normal_only_blocks"]
        + counts["outlier_only_blocks"]
        + counts["mixed_blocks"]
    ):
        raise RuntimeError(f"{domain} block partition counts are inconsistent.")
    if counts["outlier_values"] > counts["total_values"]:
        raise RuntimeError(f"{domain} outlier count exceeds total values.")
    if counts["outlier_blocks"] != (
        counts["outlier_only_blocks"] + counts["mixed_blocks"]
    ):
        raise RuntimeError(f"{domain} outlier-block counts are inconsistent.")

result = {
    "model": MODEL_ID,
    "dataset": f"{DATASET_ID}/{DATASET_CONFIG}",
    "split": SPLIT,
    "quantization": (
        f"W/A BiE{PRIVATE_VALUE_BITS} fake quantization with dynamic "
        "signed-tensor mu+3sigma thresholds"
    ),
    "method_label": "BiE signed-3sigma explicit-outlier research variant",
    "paper_relation": (
        "BiE numerical format with the Oiso T1/T3-style signed-tensor mu+3sigma "
        "explicit-outlier threshold; not the original BiE offline Bayesian threshold "
        "search or Oiso final MSE-optimized threshold reproduction"
    ),
    "format": (
        f"BiE{PRIVATE_VALUE_BITS} (1S{BIE.mantissa_bits}M + two shared "
        f"E{BIE.shared_exponent_bits} + 1-bit type, G{BIE.block_size})"
    ),
    "bie_config": asdict(BIE),
    "threshold_contract": {
        "formula": "mean(X) + sigma_k * std(X)",
        "statistics_domain": "signed tensor values",
        "comparison": "outlier iff abs(X) > threshold",
        "std_correction": 0,
        "weight_granularity": "one complete nn.Linear weight tensor",
        "activation_granularity": "one complete nn.Linear input tensor per forward call",
        "computed_before_chunking": True,
    },
    "storage_contract": {
        "private_value_bits": PRIVATE_VALUE_BITS,
        "type_bits_per_value": 1,
        "shared_exponents_per_block": 2,
        "shared_exponent_bits": BIE.shared_exponent_bits,
        "effective_bits_per_value_excluding_tensor_threshold": (
            EFFECTIVE_BITS_PER_VALUE
        ),
        "packed_storage_implemented": False,
    },
    "quantized_scope": {
        "operations": "all nn.Linear modules",
        "weights": True,
        "activations": True,
        "lm_head": BIE.quantize_lm_head,
        "attention_internal_matmul": False,
        "softmax": False,
        "layernorm": False,
    },
    "linear_output_dtype": "float16",
    "matmul_backend": "torch.nn.functional.linear with dequantized FP16 operands",
    "quantized_linear_layers": len(bie_layers),
    "attention_implementation": "eager",
    "context_length": CONTEXT_LENGTH,
    "stride": STRIDE,
    "evaluation_protocol": EVALUATION_PROTOCOL,
    "drop_remainder": DROP_REMAINDER,
    "baseline_perplexity": BASELINE_PPL,
    "delta_perplexity": metrics["perplexity"] - BASELINE_PPL,
    "quantization_stats": quantization_stats,
    "gpu": torch.cuda.get_device_name(0),
    "cuda": torch.version.cuda,
    "python": platform.python_version(),
    "pytorch": torch.__version__,
    "transformers": transformers.__version__,
    "datasets": datasets.__version__,
    **metrics,
}

OUTPUT_PATH.write_text(
    json.dumps(result, indent=2, ensure_ascii=False),
    encoding="utf-8",
)
print(json.dumps({
    "perplexity": result["perplexity"],
    "delta_perplexity": result["delta_perplexity"],
    "weight_outlier_rate": (
        result["quantization_stats"]["aggregate"]["weight"]["rates"]["outlier_value_rate"]
    ),
    "activation_outlier_rate": (
        result["quantization_stats"]["aggregate"]["activation"]["rates"]["outlier_value_rate"]
    ),
}, indent=2, ensure_ascii=False))
print(f"Saved: {OUTPUT_PATH.resolve()}")

## Download result

In [ ]:
if not OUTPUT_PATH.is_file():
    raise FileNotFoundError(f"Result JSON does not exist: {OUTPUT_PATH}")

try:
    from google.colab import files
except ImportError:
    print(f"Not running in Colab. JSON remains at: {OUTPUT_PATH.resolve()}")
else:
    files.download(str(OUTPUT_PATH))